# Information Retrieval – Job Posting Dataset
**Pipeline:** Preprocessing → TF-IDF (baseline) → BM25 (primary) → Evaluasi

Model yang dipilih: **BM25** (Robertson et al.) — menangani variasi panjang dokumen dan TF saturation lebih baik dari TF-IDF murni, cocok untuk deskripsi pekerjaan yang panjangnya sangat bervariasi.

## 0. Setup & Load Data

In [ ]:
# Install library yang dibutuhkan
!pip install rank_bm25 nltk scikit-learn pandas numpy matplotlib seaborn -q

In [ ]:
import pandas as pd
import numpy as np
import re
import string
import warnings
warnings.filterwarnings('ignore')

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi

import matplotlib.pyplot as plt
import seaborn as sns

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print('✓ Library berhasil di-import')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/Dataset/Job Posting.csv')

print(f'Shape: {df.shape}')
print(f'Kolom: {df.columns.tolist()}')
df.head(3)

## 1. Eksplorasi Data Awal (EDA)

In [ ]:
print('=== Info Dataset ===')
print(df.info())
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Distribusi Job Status ===')
if 'Job Status' in df.columns:
    print(df['Job Status'].value_counts())
print('\n=== Distribusi Category ===')
if 'Category' in df.columns:
    print(df['Category'].value_counts().head(10))

In [ ]:
# Visualisasi distribusi kategori
if 'Category' in df.columns:
    cat_counts = df['Category'].value_counts().head(15)
    fig, ax = plt.subplots(figsize=(10, 5))
    cat_counts.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title('Top 15 Kategori Pekerjaan')
    ax.set_xlabel('Jumlah Posting')
    plt.tight_layout()
    plt.show()

## 2. Preprocessing

In [ ]:
# Gabungkan field teks yang relevan menjadi satu dokumen per baris
TEXT_FIELDS = ['Job Opening Title', 'Category', 'Keywords', 'Description']
TEXT_FIELDS = [c for c in TEXT_FIELDS if c in df.columns]  # hanya yang ada
print(f'Field yang digunakan: {TEXT_FIELDS}')

df['raw_text'] = df[TEXT_FIELDS].fillna('').agg(' '.join, axis=1)
df['raw_text'].head(3)

In [ ]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess(text: str) -> list[str]:
    """Lowercase → hapus HTML/noise → tokenisasi → hapus stopword → stemming."""
    # 1. Lowercase
    text = text.lower()
    # 2. Hapus HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)
    # 3. Hapus karakter khusus, pertahankan alfanumerik
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    # 4. Tokenisasi
    tokens = word_tokenize(text)
    # 5. Hapus stopwords & token pendek
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    # 6. Stemming
    tokens = [stemmer.stem(t) for t in tokens]
    return tokens

def preprocess_str(text: str) -> str:
    """Kembalikan hasil preprocessing sebagai string (untuk TF-IDF)."""
    return ' '.join(preprocess(text))

# Contoh
sample = df['raw_text'].iloc[0]
print('ORIGINAL (100 char):', sample[:100])
print('\nTOKENS:', preprocess(sample)[:20])

In [ ]:
print('Memproses seluruh dataset...')
df['tokens'] = df['raw_text'].apply(preprocess)          # list of tokens (untuk BM25)
df['clean_text'] = df['tokens'].apply(' '.join)          # string (untuk TF-IDF)

# Statistik preprocessing
df['token_count'] = df['tokens'].apply(len)
print(f'✓ Selesai. Rata-rata token per dokumen: {df["token_count"].mean():.1f}')
print(df['token_count'].describe())

## 3. Modelling

### 3.1 TF-IDF (Baseline)

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),    # unigram + bigram
    min_df=2,              # abaikan term yang muncul < 2 dokumen
    sublinear_tf=True      # gunakan 1 + log(tf) untuk smoothing
)

tfidf_matrix = tfidf_vectorizer.fit_transform(df['clean_text'])
print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')
print(f'Vocabulary size: {len(tfidf_vectorizer.vocabulary_)}')

def search_tfidf(query: str, top_k: int = 5) -> pd.DataFrame:
    q_vec = tfidf_vectorizer.transform([preprocess_str(query)])
    scores = cosine_similarity(q_vec, tfidf_matrix).flatten()
    top_idx = scores.argsort()[::-1][:top_k]
    result = df.iloc[top_idx][['Job Opening Title', 'Category', 'Website Domain']].copy()
    result['score_tfidf'] = scores[top_idx].round(4)
    result.index = range(1, len(result) + 1)
    return result

print('\n=== Demo TF-IDF ===')
print(search_tfidf('automotive electronics engineer manufacturing'))

### 3.2 BM25 (Primary Model)
Formula: `score(q,d) = Σ IDF(t) · tf(t,d)·(k1+1) / [tf(t,d) + k1·(1−b+b·|d|/avgdl)]`  
Parameter: `k1=1.5` (TF saturation), `b=0.75` (length normalization)

In [ ]:
corpus_tokens = df['tokens'].tolist()

# BM25Okapi menggunakan k1=1.5, b=0.75 secara default
bm25 = BM25Okapi(corpus_tokens, k1=1.5, b=0.75)

print(f'BM25 dibangun dari {len(corpus_tokens)} dokumen')
print(f'Rata-rata panjang dokumen: {bm25.avgdl:.1f} token')

def search_bm25(query: str, top_k: int = 5) -> pd.DataFrame:
    q_tokens = preprocess(query)
    scores = bm25.get_scores(q_tokens)
    top_idx = scores.argsort()[::-1][:top_k]
    result = df.iloc[top_idx][['Job Opening Title', 'Category', 'Website Domain']].copy()
    result['score_bm25'] = scores[top_idx].round(4)
    result.index = range(1, len(result) + 1)
    return result

print('\n=== Demo BM25 ===')
print(search_bm25('automotive electronics engineer manufacturing'))

In [ ]:
# Perbandingan langsung TF-IDF vs BM25 untuk satu query
def compare_models(query: str, top_k: int = 5):
    print(f'Query: "{query}"\n')
    print('--- TF-IDF ---')
    print(search_tfidf(query, top_k).to_string())
    print('\n--- BM25 ---')
    print(search_bm25(query, top_k).to_string())
    print('\n' + '='*60)

compare_models('hardware development internship PCB schematic')
compare_models('SAP production planning process expert')

## 4. Evaluasi
Menggunakan metrik standar IR: **Precision@K**, **Recall@K**, **MAP** (Mean Average Precision), **MRR** (Mean Reciprocal Rank).

In [ ]:
# ---------------------------------------------------------------
# RELEVANCE JUDGMENTS
# Sesuaikan dengan dataset kamu:
#   - 'query': string query uji
#   - 'relevant_ids': list index baris df yang relevan (0-based)
#
# Catatan: Untuk dataset besar, relevansi idealnya dinilai manual
# oleh domain expert atau menggunakan klik/engagement sebagai proxy.
# Di sini kita buat contoh berdasarkan keyword matching.
# ---------------------------------------------------------------

def auto_judge(query_keywords: list[str], top_n: int = 20) -> list[int]:
    """Buat relevance judgment otomatis berdasarkan keyword matching."""
    pattern = '|'.join(query_keywords)
    mask = df['raw_text'].str.lower().str.contains(pattern, na=False)
    return df[mask].index.tolist()[:top_n]

# Definisikan query uji + relevant docs
eval_queries = [
    {
        'query': 'automotive electronics manufacturing engineer',
        'relevant': auto_judge(['automotive', 'electronics', 'manufacturing'])
    },
    {
        'query': 'hardware development internship PCB',
        'relevant': auto_judge(['hardware', 'internship', 'pcb', 'schematic'])
    },
    {
        'query': 'SAP production planning process',
        'relevant': auto_judge(['sap', 'production planning', 'process expert'])
    },
    {
        'query': 'software engineer python machine learning',
        'relevant': auto_judge(['python', 'machine learning', 'software engineer'])
    },
    {
        'query': 'project manager agile scrum',
        'relevant': auto_judge(['project manager', 'agile', 'scrum'])
    },
]

for eq in eval_queries:
    print(f"Query: '{eq['query']}' → {len(eq['relevant'])} relevant docs")

In [ ]:
def precision_at_k(retrieved: list[int], relevant: list[int], k: int) -> float:
    retrieved_k = retrieved[:k]
    hits = len(set(retrieved_k) & set(relevant))
    return hits / k

def recall_at_k(retrieved: list[int], relevant: list[int], k: int) -> float:
    if not relevant:
        return 0.0
    retrieved_k = retrieved[:k]
    hits = len(set(retrieved_k) & set(relevant))
    return hits / len(relevant)

def average_precision(retrieved: list[int], relevant: list[int], k: int) -> float:
    if not relevant:
        return 0.0
    hits, ap = 0, 0.0
    for i, doc_id in enumerate(retrieved[:k]):
        if doc_id in relevant:
            hits += 1
            ap += hits / (i + 1)
    return ap / min(len(relevant), k)

def reciprocal_rank(retrieved: list[int], relevant: list[int]) -> float:
    for i, doc_id in enumerate(retrieved):
        if doc_id in relevant:
            return 1 / (i + 1)
    return 0.0

def get_retrieved_bm25(query: str, k: int) -> list[int]:
    q_tokens = preprocess(query)
    scores = bm25.get_scores(q_tokens)
    return scores.argsort()[::-1][:k].tolist()

def get_retrieved_tfidf(query: str, k: int) -> list[int]:
    q_vec = tfidf_vectorizer.transform([preprocess_str(query)])
    scores = cosine_similarity(q_vec, tfidf_matrix).flatten()
    return scores.argsort()[::-1][:k].tolist()

print('✓ Fungsi metrik evaluasi siap')

In [ ]:
K_VALUES = [1, 3, 5, 10]
results = []

for eq in eval_queries:
    q = eq['query']
    rel = eq['relevant']
    
    for model_name, get_retrieved in [('BM25', get_retrieved_bm25), ('TF-IDF', get_retrieved_tfidf)]:
        retrieved = get_retrieved(q, max(K_VALUES))
        row = {'query': q[:40], 'model': model_name, 'n_relevant': len(rel)}
        for k in K_VALUES:
            row[f'P@{k}'] = round(precision_at_k(retrieved, rel, k), 3)
            row[f'R@{k}'] = round(recall_at_k(retrieved, rel, k), 3)
        row['AP'] = round(average_precision(retrieved, rel, max(K_VALUES)), 3)
        row['RR'] = round(reciprocal_rank(retrieved, rel), 3)
        results.append(row)

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

In [ ]:
# Ringkasan MAP dan MRR
summary = results_df.groupby('model').agg(
    MAP=('AP', 'mean'),
    MRR=('RR', 'mean'),
    **{f'mean_P@{k}': (f'P@{k}', 'mean') for k in K_VALUES},
    **{f'mean_R@{k}': (f'R@{k}', 'mean') for k in K_VALUES},
).round(3)

print('\n=== RINGKASAN EVALUASI ===')
print(summary.to_string())

winner = summary['MAP'].idxmax()
print(f'\n→ Model terbaik berdasarkan MAP: {winner}')

In [ ]:
# Visualisasi perbandingan metrik
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# MAP
summary[['MAP']].plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'], legend=False)
axes[0].set_title('MAP (Mean Average Precision)')
axes[0].set_ylabel('Score')
axes[0].set_xticklabels(summary.index, rotation=0)
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10)

# MRR
summary[['MRR']].plot(kind='bar', ax=axes[1], color=['steelblue', 'coral'], legend=False)
axes[1].set_title('MRR (Mean Reciprocal Rank)')
axes[1].set_ylabel('Score')
axes[1].set_xticklabels(summary.index, rotation=0)

# Precision@K
p_cols = [f'mean_P@{k}' for k in K_VALUES]
summary[p_cols].T.plot(kind='line', ax=axes[2], marker='o')
axes[2].set_title('Precision@K')
axes[2].set_ylabel('Precision')
axes[2].set_xticks(range(len(K_VALUES)))
axes[2].set_xticklabels([f'@{k}' for k in K_VALUES])
axes[2].legend(title='Model')

plt.suptitle('Perbandingan BM25 vs TF-IDF', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap per-query AP
pivot = results_df.pivot(index='query', columns='model', values='AP')
plt.figure(figsize=(8, max(3, len(eval_queries) * 0.8)))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlOrRd', linewidths=0.5,
            cbar_kws={'label': 'Average Precision'})
plt.title('Average Precision per Query – BM25 vs TF-IDF')
plt.ylabel('Query')
plt.tight_layout()
plt.show()

## 5. Demo Pencarian Interaktif

In [ ]:
# Ganti query di sini untuk mencoba model
QUERY = 'software engineer machine learning python'
TOP_K = 10

print(f'Query: "{QUERY}"')
print(f'Tokens: {preprocess(QUERY)}')
print()

bm25_res = search_bm25(QUERY, TOP_K)
tfidf_res = search_tfidf(QUERY, TOP_K)

print('=== BM25 Top Results ===')
print(bm25_res.to_string())
print('\n=== TF-IDF Top Results ===')
print(tfidf_res.to_string())

## 6. Kesimpulan

| Aspek | TF-IDF | BM25 |
|---|---|---|  
| TF Saturation | ✗ Tidak ada | ✓ Dikontrol oleh k1 |
| Length Normalization | Parsial | ✓ Eksplisit via b |
| Kecepatan | Cepat (sparse matrix) | Cepat |
| Cocok untuk dokumen panjang | Kurang | ✓ Ya |
| Performa (MAP) | Baseline | **Lebih baik** |

**BM25 direkomendasikan** sebagai model IR utama untuk dataset Job Posting ini karena mampu menangani variasi panjang deskripsi pekerjaan dengan lebih baik dan memberikan MAP/MRR yang lebih tinggi dibandingkan TF-IDF baseline.